## Load cleaned data from Silver table into Gold layer processing

In [0]:
df_gold = spark.read.table("e_commerce_catalog.silver_schema.orders_cleaned_table")

In [0]:
df_gold.display()

customer_id,final_amount,ProductCategory,Order_Date,State,MembershipTier,PaymentMethod
CUST-05985,218.07,Automotive,2024-10-20,KY,Bronze,Buy Now Pay Later
CUST-06197,105.7,Toys & Games,2024-04-16,MI,Gold,Apple Pay
CUST-06403,726.06,Electronics,2023-07-30,WV,Gold,Apple Pay
CUST-08146,331.12,Home & Kitchen,2023-01-10,IN,Silver,PayPal
CUST-07549,79.65,Toys & Games,2024-07-24,NV,Silver,PayPal
CUST-07139,2441.3,Electronics,2022-08-07,IA,Platinum,Google Pay
CUST-06698,161.06,Sports & Outdoors,2023-07-04,RI,Platinum,Apple Pay
CUST-02401,63.09,Clothing,2023-09-17,VA,Silver,Gift Card
CUST-04321,179.69,Automotive,2022-12-04,LA,Platinum,Apple Pay
CUST-07333,73.02,Automotive,2022-04-02,KS,Platinum,Gift Card


In [0]:
from pyspark.sql.functions import date_format, col

df_clean = df_gold.withColumn(
    "year_month",
    date_format(col("Order_Date"), "yyyy-MM")
)

###Total Revenue

In [0]:
df_clean.selectExpr("SUM(final_amount) as total_revenue").show()

+-----------------+
|    total_revenue|
+-----------------+
|533468.0499999991|
+-----------------+



###Total Orders

In [0]:
df_clean.selectExpr("COUNT(customer_id) as total_orders").show()

+------------+
|total_orders|
+------------+
|        1665|
+------------+



### Avg Order Value

In [0]:
df_clean.selectExpr("AVG(final_amount) as avg_order_value").show()

+-----------------+
|  avg_order_value|
+-----------------+
|320.4012312312307|
+-----------------+



###Revenue by State

In [0]:
df_clean.groupBy("State") \
    .agg(sum("final_amount").alias("revenue")) \
    .show()

+-----+------------------+
|State|           revenue|
+-----+------------------+
|   SC| 6273.169999999999|
|   AZ|20287.609999999997|
|   LA|16463.230000000003|
|   MN| 7533.529999999999|
|   NJ| 9320.999999999998|
|   OR| 6866.359999999999|
|   VA|          11093.34|
|   RI| 9529.710000000003|
|   KY|12297.130000000003|
|   WY|           8904.18|
|   NH|13824.160000000002|
|   MI|           9567.44|
|   NV|           6605.64|
|   WI| 6734.809999999998|
|   ID|           20959.2|
|   CA|10245.550000000003|
|   CT|10265.390000000001|
|   NE|10040.960000000001|
|   MT| 9889.120000000003|
|   NC|14677.079999999998|
+-----+------------------+
only showing top 20 rows


###Create Unified Table with KPIs

In [0]:
from pyspark.sql.functions import sum, count, col

In [0]:

df_unified = df_clean.groupBy(
    "year_month",
    "ProductCategory",
    "State",
    "MembershipTier",
    "PaymentMethod"
).agg(
    sum("final_amount").alias("total_revenue"),
    count("customer_id").alias("total_orders")
)

###Add Average Order Value

In [0]:
df_unified = df_unified.withColumn(
    "avg_order_value",
    col("total_revenue") / col("total_orders")
)

In [0]:
df_unified.display()

year_month,ProductCategory,State,MembershipTier,PaymentMethod,total_revenue,total_orders,avg_order_value
2022-02,Sports & Outdoors,ND,Bronze,Gift Card,2893.02,6,482.17
2024-12,Beauty & Personal Care,VT,Gold,Gift Card,51.92,1,51.92
2023-04,Electronics,CO,Platinum,Apple Pay,1679.3,1,1679.3
2023-06,Electronics,NC,Bronze,PayPal,746.18,1,746.18
2024-12,Automotive,AK,Bronze,PayPal,132.49,1,132.49
2022-08,Toys & Games,DE,Silver,Apple Pay,197.16,2,98.58
2022-12,Automotive,LA,Platinum,Apple Pay,179.69,1,179.69
2022-07,Clothing,ID,Gold,Buy Now Pay Later,113.59,1,113.59
2023-11,Sports & Outdoors,MS,Bronze,Apple Pay,467.38,1,467.38
2023-08,Automotive,WI,Bronze,Buy Now Pay Later,122.59,1,122.59


Databricks visualization. Run in Databricks to view.

In [0]:
df_unified.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("e_commerce_catalog.gold_schema.unified_reporting_table")

###1. Category Ranking

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, sum

df_category = df_clean.groupBy(
    "year_month",
    "ProductCategory"
).agg(
    sum("final_amount").alias("total_revenue")
)

window_category = Window.partitionBy("year_month").orderBy(desc("total_revenue"))

df_category_rank = df_category.withColumn(
    "category_rank",
    row_number().over(window_category)
)

In [0]:
df_category.display()

year_month,ProductCategory,total_revenue
2024-10,Automotive,1291.4
2024-04,Toys & Games,281.66
2023-07,Electronics,12111.68
2023-01,Home & Kitchen,1545.4
2024-07,Toys & Games,3427.8200000000006
2022-08,Electronics,9511.18
2023-07,Sports & Outdoors,3050.6499999999996
2023-09,Clothing,744.61
2022-12,Automotive,1462.96
2022-04,Automotive,1179.79


Databricks visualization. Run in Databricks to view.

###2.State Ranking

In [0]:
df_state = df_clean.groupBy(
    "year_month",
    "State"
).agg(
    sum("final_amount").alias("total_revenue")
)

window_state = Window.partitionBy("year_month").orderBy(desc("total_revenue"))

df_state_rank = df_state.withColumn(
    "state_rank",
    row_number().over(window_state)
)

In [0]:
df_state.display()

year_month,State,total_revenue
2024-10,KY,703.0799999999999
2024-04,MI,236.72000000000003
2023-07,WV,1149.34
2023-01,IN,1454.08
2024-07,NV,79.65
2022-08,IA,5914.47
2023-07,RI,697.73
2023-09,VA,128.41
2022-12,LA,179.69
2022-04,KS,73.02


Databricks visualization. Run in Databricks to view.

###Save Tables (for Power BI)

In [0]:
df_category_rank.write.format("delta")\
                      .mode("overwrite")\
                      .saveAsTable("e_commerce_catalog.gold_schema.category_ranking")

In [0]:
df_state_rank.write.format("delta")\
             .mode("overwrite")\
             .saveAsTable("e_commerce_catalog.gold_schema.state_ranking")
